# Blind politeness review

Open the [25-opening review sheet](../../../data/processed/linguistic-features/politeness-experiment/phase-4/20261124T055238584162Z/blind_politeness_review.csv). For each row, read the whole opening and fill in `manual_label`. Use `notes` for mixed tone, sarcasm, quoted language, or places where missing context makes a judgment difficult. Each row comes from a different discussion. Review the text before running the agreement analysis.

| Enter in `manual_label` | How to use it |
| --- | --- |
| `impolite` | Dismissive, condescending, mocking, or openly hostile toward the person or their contribution. |
| `neutral` | Straightforward or factual delivery with little visible warmth, courtesy, or disrespect. Direct disagreement can be neutral. |
| `somewhat polite` | Respectful and considerate in wording, with limited warmth or explicit courtesy. |
| `polite` | Clearly considerate and respectful in tone, often with warmth, acknowledgment, or courteous phrasing. |
| `unclear` | There is too little context or the tone is too ambiguous to assign a label. |

Base the label on how the reply addresses people and their views. Assess the author’s own wording and intended tone. Judge the overall reply when it mixes tones and record the relevant passage in `notes`. Save the CSV with the original `review_id` values.

## Agreement after review

Run the cells below once you have added your labels. Spearman correlation compares human ratings with the continuous model scores and the courtesy count. Quadratic weighted kappa, mean absolute ordinal error, and confusion matrices compare human ratings with each model’s predicted class. Blank and unclear ratings are excluded.

The 25 openings were deliberately selected for variety. Agreement estimates describe this review sample and are exploratory.

In [1]:
from pathlib import Path
import json

import pandas as pd

project = next(
    path / "project" for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "team-setup").is_dir()
)
run_id = "20261124T055238584162Z"
run = project / "data/processed/linguistic-features/politeness-experiment/phase-4" / run_id

review_csv = run / "blind_politeness_review.csv"
ratings = pd.read_csv(review_csv, dtype=str, keep_default_na=False, usecols=["review_id", "manual_label"])
labels = {"impolite": 0, "neutral": 1, "somewhat polite": 2, "polite": 3}
rating = ratings["manual_label"].str.strip().str.casefold()
unknown = sorted(set(rating) - {*labels, "", "unclear"})
if unknown:
    raise ValueError(f"Unknown labels: {unknown}")
rated = ratings.assign(human=rating.map(labels)).loc[rating.isin(labels), ["review_id", "human"]].copy()
print(f"{len(rated)}/{len(ratings)} usable ratings; {rating.eq('unclear').sum()} unclear.")
if rated.empty:
    print("Fill in the blind review CSV before running the agreement analysis.")


0/25 usable ratings; 0 unclear.
Fill in the blind review CSV before running the agreement analysis.


In [2]:

if not rated.empty:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix

    key = pd.read_parquet(run / "review_key.parquet")
    if not ratings["review_id"].is_unique or set(ratings["review_id"]) != set(key["review_id"]):
        raise ValueError("Keep each original review ID exactly once.")
    review = rated.merge(key, on="review_id", validate="one_to_one")
    human = review["human"]
    records, matrices = [], {}
    for size in (128, 256, 510):
        predicted = review[f"class_{size}"].map(labels)
        matrices[size] = pd.DataFrame(
            confusion_matrix(human, predicted, labels=list(labels.values())),
            index=pd.Index(labels, name="Human"), columns=pd.Index(labels, name="Model"),
        )
        kappa = None
        if len(review) > 1 and pd.concat([human, predicted]).nunique() > 1:
            kappa = cohen_kappa_score(human, predicted, labels=list(labels.values()), weights="quadratic")
        records.append({
            "Measure": f"Polite Guard {size}", "Spearman vs human": human.rank().corr(review[f"score_{size}"].rank()),
            "Quadratic kappa": kappa, "Mean absolute class error": (human - predicted).abs().mean(),
        })
    records.append({
        "Measure": "ConvoKit courtesy count", "Spearman vs human": human.rank().corr(review["courtesy_count"].rank()),
    })
    print(pd.DataFrame(records).set_index("Measure").round(3).to_string())
    for size, matrix in matrices.items():
        print(f"\nPolite Guard {size}")
        print(matrix.to_string())
